 SECOND ADVANCED -  FEATURE EXTRACTOR_final

In [1]:


import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from scipy.stats import skew
from skimage.feature import (graycomatrix,graycoprops,local_binary_pattern)
from skimage.measure import shannon_entropy


def extract_edge_features(image):

    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)

   
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

    magnitude = np.sqrt(sobelx**2 + sobely**2)
    edge_mean = np.mean(magnitude)
    edge_std = np.std(magnitude)
    edge_energy = np.sum(magnitude**2) / (gray.shape[0] * gray.shape[1])

    edge_skew = skew(magnitude.flatten())
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    lap_var = np.var(lap)
    return [edge_mean,edge_std,edge_energy,edge_skew,lap_var]


def extract_color_features_hsv(image):
    image_np = np.array(image)
    hsv = cv2.cvtColor(image_np, cv2.COLOR_RGB2HSV)
    features = []
    channel_ranges = [(0, 180),(0, 256),(0, 256)]

    for i, channel_range in enumerate(channel_ranges):

        channel = hsv[:, :, i]
        hist = np.histogram(channel, bins=16,range=channel_range)[0].astype(np.float32)
        hist = hist / (np.sum(hist) + 1e-7)

      
        features.extend(hist.tolist())

        
        features.append(np.mean(channel))
        features.append(np.std(channel))
        features.append(skew(channel.flatten()))

    return features


def extract_color_features_lab(image):
    image_np = np.array(image)
    lab = cv2.cvtColor(image_np, cv2.COLOR_RGB2LAB)
    features = []

    for i in range(3):

        channel = lab[:, :, i]

        features.append(np.mean(channel))
        features.append(np.std(channel))
        features.append(skew(channel.flatten()))

    return features


def extract_texture_features(image):
    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)
    gray_reduced = gray // 8
    glcm = graycomatrix(
        gray_reduced,
        distances=[1],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=32,
        symmetric=True,
        normed=True)

    contrast = graycoprops(glcm,'contrast').mean()
    homogeneity = graycoprops(glcm,'homogeneity').mean()
    correlation = graycoprops(glcm,'correlation').mean()
    energy = graycoprops(glcm,'energy').mean()
    dissimilarity = graycoprops(glcm,'dissimilarity').mean()
    entropy = shannon_entropy(gray)
    
    return [contrast,
        homogeneity,
        correlation,
        energy,
        dissimilarity,
        entropy]

def extract_lbp_features(image):
    gray = cv2.cvtColor(np.array(image),cv2.COLOR_RGB2GRAY)
    radius = 2
    n_points = 8 * radius
    lbp = local_binary_pattern(gray,n_points,radius,method="uniform")
    hist, _ = np.histogram(lbp.ravel(),bins=np.arange(0, n_points + 3),range=(0, n_points + 2))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-7)
    return hist.tolist()


def extract_all_features(image):
    edge_features = extract_edge_features(image)
    hsv_features = extract_color_features_hsv(image)
    lab_features = extract_color_features_lab(image)
    texture_features = extract_texture_features(image)
    lbp_features = extract_lbp_features(image)

    return (
        edge_features +
        hsv_features +
        lab_features +
        texture_features +
        lbp_features)

train_path = "/kaggle/input/datasets/project5sanju/normal-data/RESIZED ALL-20260522T045904Z-3-001/RESIZED ALL"
data = []

for class_name in os.listdir(train_path):
    class_path = os.path.join(train_path, class_name)

    if not os.path.isdir(class_path):
        continue
    print(f"Processing: {class_name}")
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)

        try:
            image = Image.open(img_path).convert("RGB")
            features = extract_all_features(image)
            row = [img_name, class_name] + features
            data.append(row)

        except Exception as e:
            print(f"Error processing {img_name}: {e}")


columns = []
columns += ["image_name"]
columns += ["label"]
columns += [
    "edge_mean",
    "edge_std",
    "edge_energy",
    "edge_skew",
    "laplacian_variance"]
columns += [f"hsv_hist_{i}" for i in range(48)]
columns += [
    "h_mean", "h_std", "h_skew",
    "s_mean", "s_std", "s_skew",
    "v_mean", "v_std", "v_skew"]
columns += [
    "lab_l_mean",
    "lab_l_std",
    "lab_l_skew",
    "lab_a_mean",
    "lab_a_std",
    "lab_a_skew",
    "lab_b_mean",
    "lab_b_std",
    "lab_b_skew"]
columns += [
    "contrast",
    "homogeneity",
    "correlation",
    "energy",
    "dissimilarity",
    "entropy"]

radius = 2
n_points = 8 * radius
columns += [f"lbp_{i}"for i in range(n_points + 2)]


df = pd.DataFrame(data, columns=columns)
print(df.head())
print("\nTotal Features:", len(columns) - 2)


save_dir = "/kaggle/working/FEATURES"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir,"SECOND_Advanced_Skin_Features_Optimized.csv")
df.to_csv(save_path, index=False)
print("\nFeature extraction completed successfully!")
print(f"Saved to: {save_path}")

Processing: MEL
Processing: VASC
Processing: DF
Processing: NV
Processing: BKL
Processing: AKIEC
Processing: BCC
               image_name label  edge_mean   edge_std  edge_energy  edge_skew  \
0  train_ISIC_0029574.jpg   MEL  41.734184  43.246410  3611.994021   4.759791   
1  train_ISIC_0033807.jpg   MEL  31.704719  21.704810  1476.287946   1.893580   
2  train_ISIC_0026993.jpg   MEL  26.192064  37.676108  2105.513353   3.646921   
3  train_ISIC_0031550.jpg   MEL  26.239672  24.594087  1293.389509   3.513425   
4  train_ISIC_0032533.jpg   MEL  20.630797  15.267722   658.733099   2.344557   

   laplacian_variance  hsv_hist_0  hsv_hist_1  hsv_hist_2  ...     lbp_8  \
0          229.453873    0.912528    0.000020    0.000000  ...  0.078962   
1           84.162942    0.000000    0.000000    0.000000  ...  0.057179   
2           53.437674    0.105529    0.006916    0.000040  ...  0.093072   
3           99.287542    0.216458    0.000000    0.000000  ...  0.054468   
4           35.75712

svm_with files_final


In [2]:


import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import (train_test_split,GridSearchCV,StratifiedKFold)
from sklearn.metrics import (classification_report,confusion_matrix,accuracy_score,f1_score,precision_score,recall_score)
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold

output_folder = "/kaggle/working/svm_results_final"
os.makedirs(output_folder, exist_ok=True)
df = pd.read_csv("/kaggle/input/datasets/project5sanju/features-for-svm/SECOND_Advanced_Skin_Features_Optimized.csv")


train_image_names = df["image_name"].values
X = df.drop(columns=["label", "image_name"]).values
y = df["label"].values
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Classes:", le.classes_)


X_train = X
y_train = y_encoded

print(f"Training samples: {X_train.shape[0]}")

pipeline = Pipeline([("scaler", StandardScaler()),("var", VarianceThreshold(threshold=0.01)),("pca", PCA(n_components=0.95)),
                     ("svm", SVC(kernel="rbf",probability=True,class_weight="balanced",random_state=42))])

param_grid = {"svm__C": [9],"svm__gamma": [0.005]}
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
grid_search = GridSearchCV(pipeline,param_grid,cv=cv,scoring="f1_weighted",n_jobs=-1,verbose=2,return_train_score=True)
grid_search.fit(X_train, y_train)
print("\nBest parameters:", grid_search.best_params_)
print("Best cross-val F1:",round(grid_search.best_score_, 4))

best_model = grid_search.best_estimator_

joblib.dump(best_model, "svm_baseline.pkl")
joblib.dump(le, "label_encoder_svm.pkl")
print("\nModel saved to svm_baseline.pkl")

train_pred = best_model.predict(X_train)
train_true = y_train
print("\n── Training report ──")
print(classification_report(train_true,train_pred,target_names=le.classes_))


train_cm = confusion_matrix(train_true,train_pred)

train_cm_df = pd.DataFrame(train_cm,index=[f"True_{name}" for name in le.classes_],
    columns=[f"Pred_{name}" for name in le.classes_])

print("\n===== SVM – Training Set =====\n")
print(train_cm_df)


df_external = pd.read_csv("/kaggle/input/datasets/project5sanju/features-for-svm/TEST_SECOND_Advanced_Skin_Features_Optimized.csv")


test_image_names = df_external["image_name"].values
X_external = df_external.drop(columns=["label", "image_name"]).values
y_external = df_external["label"].values
all_labels = le.transform(y_external)
all_preds = best_model.predict(X_external)

print("\n── External test report ──")
print(classification_report(all_labels,all_preds,target_names=le.classes_))
print("Accuracy:",round(accuracy_score(all_labels, all_preds),4))
print("F1 (weighted):",round(f1_score(all_labels,all_preds,average="weighted"),4))



test_cm = confusion_matrix(all_labels,all_preds)
test_cm_df = pd.DataFrame(test_cm,index=[f"True_{name}" for name in le.classes_],
    columns=[f"Pred_{name}" for name in le.classes_])

print("\n===== SVM – External Test Set =====\n")
print(test_cm_df)



predictions_df = pd.DataFrame({
    "image_name": test_image_names,
    "true_label": all_labels,
    "predicted_label": all_preds})

predictions_df.to_csv(os.path.join(output_folder, "predictions.csv"),index=False)


train_report = classification_report(train_true,train_pred,target_names=le.classes_,output_dict=True)
train_report_df = pd.DataFrame(train_report).transpose()
train_report_df.to_csv(os.path.join(output_folder,"train_classification_report.csv"))



test_report = classification_report(all_labels,all_preds,target_names=le.classes_,output_dict=True)
test_report_df = pd.DataFrame(test_report).transpose()
test_report_df.to_csv(os.path.join(output_folder,"test_classification_report.csv"))


train_cm_df.to_csv(os.path.join(output_folder,"train_confusion_matrix.csv"))

test_cm_df.to_csv(os.path.join(output_folder,"test_confusion_matrix.csv"))



summary_df = pd.DataFrame({"metric": ["best_crossval_f1","test_accuracy","weighted_precision","weighted_recall","weighted_f1_score"],

    "value": [grid_search.best_score_,accuracy_score(all_labels,all_preds),
        precision_score(all_labels,all_preds,average="weighted"),
        recall_score(all_labels,all_preds, average="weighted"),
        f1_score(all_labels,all_preds,average="weighted")]})

summary_df.to_csv(os.path.join(output_folder,"summary_metrics.csv"),index=False)


train_class_counts = np.bincount(train_true)
test_class_counts = np.bincount(all_labels)

class_distribution_df = pd.DataFrame({
    "class_name": le.classes_,
    "train_count": train_class_counts,
    "test_count": test_class_counts})

class_distribution_df.to_csv(os.path.join(output_folder,"class_distribution.csv"),index=False)


split_df = pd.DataFrame({
    "split": ["train","test"],
    "count": [len(train_true),len(all_labels)]})

split_df.to_csv(os.path.join(output_folder,"dataset_split_counts.csv"),index=False)


cv_results_df = pd.DataFrame(grid_search.cv_results_)
cv_results_df.to_csv(os.path.join(output_folder,"cross_validation_results.csv"),index=False)



train_predictions_df = pd.DataFrame({"image_name": train_image_names,"true_label": train_true,"predicted_label": train_pred})
train_predictions_df.to_csv(os.path.join(output_folder,"train_predictions.csv"),index=False)



print("\nALL SVM FILES SAVED SUCCESSFULLY")
print("\nSaved files:\n")
for file in os.listdir(output_folder):
    print(file)

Classes: ['AKIEC' 'BCC' 'BKL' 'DF' 'MEL' 'NV' 'VASC']
Training samples: 8427
Fitting 5 folds for each of 1 candidates, totalling 5 fits

Best parameters: {'svm__C': 9, 'svm__gamma': 0.005}
Best cross-val F1: 0.7113

Model saved to svm_baseline.pkl

── Training report ──
              precision    recall  f1-score   support

       AKIEC       0.89      0.87      0.88       655
         BCC       0.53      0.90      0.67       410
         BKL       0.56      0.71      0.62       875
          DF       0.62      0.93      0.74       184
         MEL       0.45      0.80      0.57       881
          NV       0.99      0.74      0.84      5220
        VASC       0.90      0.99      0.94       202

    accuracy                           0.77      8427
   macro avg       0.70      0.85      0.75      8427
weighted avg       0.85      0.77      0.79      8427


===== SVM – Training Set =====

            Pred_AKIEC  Pred_BCC  Pred_BKL  Pred_DF  Pred_MEL  Pred_NV  \
True_AKIEC         570   

In [4]:
import shutil
shutil.make_archive(
    "/kaggle/working/svm_results_final",'zip',
    "/kaggle/working/svm_results_final")
print("ZIP file created!")

ZIP file created!
[CV] END .........................svm__C=9, svm__gamma=0.005; total time=  25.1s
[CV] END .........................svm__C=9, svm__gamma=0.005; total time=  25.1s
[CV] END .........................svm__C=9, svm__gamma=0.005; total time=  25.4s
[CV] END .........................svm__C=9, svm__gamma=0.005; total time=  24.7s
[CV] END .........................svm__C=9, svm__gamma=0.005; total time=  14.7s
